Hugging Face transformers 라이브러리를 사용하여 문서 요약 모델을 구현하는 미션입니다. 데이터 로드 및 전처리부터 요약 모델 실행, 결과 평가까지 전체 파이프라인을 구축해 보세요.

In [12]:
import os

ROOT_DIR = os.getcwd()

# 코랩 모드
if ROOT_DIR == "/content":
    pass
    # print("[[ colab ]]")
    
    # import unicodedata
    
    # DATA_DIR = os.path.join(ROOT_DIR, "data")
    # TEXT_DIR = os.path.join(ROOT_DIR, "raw")

    # if "raw.tar.gz" not in os.listdir():
    #     # subprocess()
    #     !wget https://github.com/wonbywondev/ML-DL/releases/download/data-v3/raw.tar.gz
    # else:
    #     print("· raw.tar.gz (O)")


    # if not os.path.exists(TEXT_DIR):
    #     # !tar -xzvf raw.tar.gz -C /content
    # else:
    #     print("· data (O)")


    # if not os.path.exists(DATA_DIR):
    #     os.mkdir(DATA_DIR)


    # train_json_path = os.path.join(TEXT_DIR, "일상생활및구어체_한영_train_set.json")
    # val_json_path = os.path.join(TEXT_DIR, "일상생활및구어체_한영_valid_set.json")

    # train_json_path = unicodedata.normalize("NFC", train_json_path)
    # val_json_path = unicodedata.normalize("NFC", val_json_path)

# 로컬 모드
else:
    print("[[ local ]]")
    ROOT_DIR = "/".join(ROOT_DIR.split("/")[:-1])
    DATA_DIR = os.path.join(ROOT_DIR, "data")
    RAW_DIR = os.path.join(DATA_DIR, "raw")

    train_edit_json_path = os.path.join(RAW_DIR, "train_original_editorial.json")
    train_law_json_path = os.path.join(RAW_DIR, "train_original_news.json")
    train_news_json_path = os.path.join(RAW_DIR, "train_original_law.json")
    val_edit_json_path = os.path.join(RAW_DIR, "valid_original_editorial.json")
    val_law_json_path = os.path.join(RAW_DIR, "valid_original_news.json")
    val_news_json_path = os.path.join(RAW_DIR, "valid_original_law.json")

[[ local ]]


In [21]:
RAW_DIR

'/Users/won/dev/00_codeit/0_mission/12_DL_transformers/data/raw'

In [13]:
# 기타 환경 설정
import torch
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import gc


# 시각화 관련 설정
try:
    plt.rcParams['font.family'] = 'Apple SD Gothic Neo'
except:
    try:
        plt.rcParams['font.family'] = 'NanumGothic'
    except:
        plt.rcParams['font.family'] = 'AppleGothic'

plt.rcParams['axes.unicode_minus'] = False
fm._load_fontmanager(try_read_cache=False)


# 디바이스 설정
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = torch.device("mps") # 맥 GPU
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda") # 윈도우 GPU
else:
    DEVICE = torch.device("cpu") # CPU


# 캐시 지우기 함수 생성
def clean_cache():
    gc.collect()
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()


# MallocStackLogging 에러 출력 방지
os.environ.pop("MallocStackLogging", None)
os.environ.pop("MallocStackLoggingNoCompact", None)
os.environ.pop("DYLD_INSERT_LIBRARIES", None)


In [14]:
import json

with open(train_edit_json_path, "r", encoding="utf-8") as f:
    raw_train_edit = json.load(f)

In [15]:
raw_train_edit.keys()

dict_keys(['name', 'delivery_date', 'documents'])

In [16]:
def dig(data):
    if type(data) == dict:
        print("dict")
        for key, value in data.items():
            print(f"key: {key}") 
            print(f"value type: {type(value)}")
            print()
            dig(value)

    elif type(data) == list:
        print("list")
        print(f"data[0]: {data[0]}")
        print(f"data[0] type: {type(data[0])}")
        print()
        dig(data[0])

    else: 
        print("instance")
        print(f"value: {data}")
        print()

dig(raw_train_edit)

dict
key: name
value type: <class 'str'>

instance
value: 사설/잡지 문서 프로젝트

key: delivery_date
value type: <class 'str'>

instance
value: 2020-12-23 15:00:19

key: documents
value type: <class 'list'>

list
data[0]: {'id': '100062073', 'category': '오피니언', 'media_type': 'online', 'media_sub_type': '경제지', 'media_name': '매일경제', 'size': 'medium', 'char_count': '1153', 'publish_date': '2011-09-01 00:03:01', 'title': '[사설] 기부문화 확산 은근히 강조한 李대통령', 'text': [[{'index': 0, 'sentence': '이명박 대통령이 어제 30대 그룹 총수를 모아놓고 "시대적 요구는 역시 총수가 앞장서야 한다. 이미 상당한 변화의 조짐이 있다는 것을 고맙게 생각한다. 총수들께서 직접 관심을 가져주시면 빨리 전파돼 긍정적인 평가를 받을 수 있다고 본다"고 말했다.', 'highlight_indices': '9,11;37,39;53,55;91,93;104,106'}, {'index': 1, 'sentence': "언뜻 보아 무슨 말인지 불분명하나 이 대통령이 지난 8ㆍ15 연설 후 정몽준 의원, 정몽구 현대차 회장이 각각 2000억원과 5000억원을 기부한 사실과 '공생발전'이란 화두를 연결하면 금방 짐작이 간다.", 'highlight_indices': '0,2;6,8;14,16;59,61;104,106'}, {'index': 2, 'sentence': '다른 그룹 총수들도 좀 나서라고 은근히 떠민 것이다.', 'highlight_indices': '0,2;11,12;18,21'}, {'index': 3, 'sentence': '이 대통령

In [17]:
from collections.abc import Mapping, Sequence

def dig(data, path="root", level=0, max_items=1, seen=None):
    """Recursively log the structure/content of nested dict/list-like data."""
    if seen is None:
        seen = set()
    indent = "· "+"  " * level
    obj_id = id(data)
    if obj_id in seen:
        print(f"{indent}{path}: <already visited>")
        return
    seen.add(obj_id)

    if isinstance(data, Mapping):
        print(f"{indent}{path}: dict ({len(data)})")
        # for key, value in list(data.items())[:max_items]:
        for key, value in list(data.items()):
            dig(value, f"{path}[{repr(key)}]", level + 1, max_items, seen)
        # if len(data) > max_items:
        #     print(f"{indent}  ... {len(data) - max_items} more keys")
    elif isinstance(data, Sequence) and not isinstance(data, (str, bytes, bytearray)):
        print(f"{indent}{path}: list ({len(data)})")
        for idx, value in enumerate(data[:max_items]):
            dig(value, f"{path}[{idx}]", level + 1, max_items, seen)
        if len(data) > max_items:
            print(f"{indent}  ... {len(data) - max_items} more items")
    else:
        print(f"{indent}{path}: {type(data).__name__} -> {repr(data)}")

dig(raw_train_edit)


· root: dict (3)
·   root['name']: str -> '사설/잡지 문서 프로젝트'
·   root['delivery_date']: str -> '2020-12-23 15:00:19'
·   root['documents']: list (56760)
·     root['documents'][0]: dict (14)
·       root['documents'][0]['id']: str -> '100062073'
·       root['documents'][0]['category']: str -> '오피니언'
·       root['documents'][0]['media_type']: str -> 'online'
·       root['documents'][0]['media_sub_type']: str -> '경제지'
·       root['documents'][0]['media_name']: str -> '매일경제'
·       root['documents'][0]['size']: str -> 'medium'
·       root['documents'][0]['char_count']: str -> '1153'
·       root['documents'][0]['publish_date']: str -> '2011-09-01 00:03:01'
·       root['documents'][0]['title']: str -> '[사설] 기부문화 확산 은근히 강조한 李대통령'
·       root['documents'][0]['text']: list (5)
·         root['documents'][0]['text'][0]: list (4)
·           root['documents'][0]['text'][0][0]: dict (3)
·             root['documents'][0]['text'][0][0]['index']: int -> 0
·             root['documents'][0]['t

In [23]:
from collections.abc import Mapping, Sequence
from pathlib import Path
import json

def iter_leaves(node, prefix=()):
    if isinstance(node, Mapping):
        if not node:
            yield prefix, None
        else:
            for key, value in node.items():
                yield from iter_leaves(value, prefix + (key,))
    elif isinstance(node, Sequence) and not isinstance(node, (str, bytes, bytearray)):
        if not node:
            yield prefix + ('__empty__',), None
        else:
            for idx, value in enumerate(node):
                yield from iter_leaves(value, prefix + (idx,))
    else:
        yield prefix, node

def flatten(node):
    return {".".join(map(str, path)): value for path, value in iter_leaves(node)}

raw = json.loads(Path(train_edit_json_path).read_text())
meta = {f"dataset.{k}": v for k, v in raw.items() if k != "documents"}
rows = [{**meta, **flatten(doc)} for doc in raw["documents"]]


In [28]:
import pandas as pd

a = pd.DataFrame(rows)

print(list(a.columns))

['dataset.name', 'dataset.delivery_date', 'id', 'category', 'media_type', 'media_sub_type', 'media_name', 'size', 'char_count', 'publish_date', 'title', 'text.0.0.index', 'text.0.0.sentence', 'text.0.0.highlight_indices', 'text.0.1.index', 'text.0.1.sentence', 'text.0.1.highlight_indices', 'text.0.2.index', 'text.0.2.sentence', 'text.0.2.highlight_indices', 'text.0.3.index', 'text.0.3.sentence', 'text.0.3.highlight_indices', 'text.1.0.index', 'text.1.0.sentence', 'text.1.0.highlight_indices', 'text.1.1.index', 'text.1.1.sentence', 'text.1.1.highlight_indices', 'text.1.2.index', 'text.1.2.sentence', 'text.1.2.highlight_indices', 'text.2.0.index', 'text.2.0.sentence', 'text.2.0.highlight_indices', 'text.2.1.index', 'text.2.1.sentence', 'text.2.1.highlight_indices', 'text.3.0.index', 'text.3.0.sentence', 'text.3.0.highlight_indices', 'text.3.1.index', 'text.3.1.sentence', 'text.3.1.highlight_indices', 'text.3.2.index', 'text.3.2.sentence', 'text.3.2.highlight_indices', 'text.4.0.index', '

In [30]:
from collections.abc import Mapping, Sequence

def dig(data, path="root", level=0, max_items=3, seen=None):
    if seen is None:
        seen = set()
    indent = "  " * level
    obj_id = id(data)
    if obj_id in seen:
        print(f"{indent}{path}: <cycle>")
        return
    seen.add(obj_id)

    if isinstance(data, Mapping):
        print(f"{indent}{path}: dict ({len(data)})")
        for key, value in list(data.items())[:max_items]:
            dig(value, f"{path}.{key}", level + 1, max_items, seen)
        if len(data) > max_items:
            print(f"{indent}  ... {len(data) - max_items} more keys")
    elif isinstance(data, Sequence) and not isinstance(data, (str, bytes, bytearray)):
        print(f"{indent}{path}: list ({len(data)})")
        for idx, value in enumerate(data[:max_items]):
            dig(value, f"{path}[{idx}]", level + 1, max_items, seen)
        if len(data) > max_items:
            print(f"{indent}  ... {len(data) - max_items} more items")
    else:
        # stop here: just show the path to this scalar/value
        print(f"{indent}{path}: -> {type(data).__name__}")


dig(raw_train_edit)

root: dict (3)
  root.name: -> str
  root.delivery_date: -> str
  root.documents: list (56760)
    root.documents[0]: dict (14)
      root.documents[0].id: -> str
      root.documents[0].category: -> str
      root.documents[0].media_type: -> str
      ... 11 more keys
    root.documents[1]: dict (14)
      root.documents[1].id: -> str
      root.documents[1].category: -> str
      root.documents[1].media_type: -> str
      ... 11 more keys
    root.documents[2]: dict (14)
      root.documents[2].id: -> str
      root.documents[2].category: -> str
      root.documents[2].media_type: -> str
      ... 11 more keys
    ... 56757 more items


In [32]:
from collections.abc import Mapping, Sequence
import json
from pathlib import Path

def dig(data, path="root", level=0, seen=None):
    if seen is None:
        seen = set()
    indent = "  " * level
    if isinstance(data, (Mapping, Sequence)) and not isinstance(data, (str, bytes, bytearray)):
        obj_id = id(data)
        if obj_id in seen:
            print(f"{indent}{path}: <cycle>")
            return
        seen.add(obj_id)

    if isinstance(data, Mapping):
        print(f"{indent}{path}: dict ({len(data)})")
        for key, value in data.items():
            next_path = f"{path}.{key}" if path else str(key)
            dig(value, next_path, level + 1, seen)
    elif isinstance(data, Sequence) and not isinstance(data, (str, bytes, bytearray)):
        print(f"{indent}{path}: list ({len(data)})")
        if data:
            dig(data[0], f"{path}[0]", level + 1, seen)
        else:
            print(f"{indent}  {path}[empty]: -> empty")
    else:
        print(f"{indent}{path}: -> {type(data).__name__}")

raw_train_edit = json.loads(Path(train_edit_json_path).read_text())
dig(raw_train_edit)


root: dict (3)
  root.name: -> str
  root.delivery_date: -> str
  root.documents: list (56760)
    root.documents[0]: dict (14)
      root.documents[0].id: -> str
      root.documents[0].category: -> str
      root.documents[0].media_type: -> str
      root.documents[0].media_sub_type: -> str
      root.documents[0].media_name: -> str
      root.documents[0].size: -> str
      root.documents[0].char_count: -> str
      root.documents[0].publish_date: -> str
      root.documents[0].title: -> str
      root.documents[0].text: list (5)
        root.documents[0].text[0]: list (4)
          root.documents[0].text[0][0]: dict (3)
            root.documents[0].text[0][0].index: -> int
            root.documents[0].text[0][0].sentence: -> str
            root.documents[0].text[0][0].highlight_indices: -> str
      root.documents[0].annotator_id: -> int
      root.documents[0].document_quality_scores: dict (4)
        root.documents[0].document_quality_scores.readable: -> int
        root.docum